# Módulo de Ingesta de Datos — Sistema de Predicción Bursátil Inteligente
### Tickers mineros peruanos y afines

**Objetivo:** descargar datos reales de mercado (OHLCV) de 5 tickers del sector minero desde Yahoo Finance, calcular indicadores técnicos (SMA, EMA, RSI) y persistirlos en MongoDB Atlas, en la colección `precios_ohlcv`.

**Tickers del proyecto:**
- `FSM` — Fortuna Silver Mines Inc.
- `VOLCABC1.LM` — Volcan Compañía Minera S.A.A.
- `ABX.TO` — Barrick Gold Corporation
- `BVN` — Compañía de Minas Buenaventura
- `BHP` — BHP Group Limited

**Fuente de datos:** Yahoo Finance vía `yfinance` (datos reales, `period='1y'`, `auto_adjust=True`).

**Persistencia:** MongoDB Atlas vía `pymongo`.

**Prerequisito:** definir el secreto `MONGO_URI` en Colab (ícono de llave 🔑 en el panel izquierdo) con la cadena de conexión a tu clúster de MongoDB Atlas.

In [1]:
# Paso 1 — Instalar librerias necesarias
!pip install "pymongo[srv]" yfinance --quiet
print("Librerias instaladas correctamente")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.5 MB/s eta 0:00:00
Librerias instaladas correctamente


In [ ]:
# Paso 2 — Conectar a MongoDB Atlas usando el secreto MONGO_URI de Colab
from google.colab import userdata
from pymongo import MongoClient

MONGO_URI = userdata.get("MONGO_URI")  # Debe estar definido en Secrets (icono de llave)
client = MongoClient(MONGO_URI)
db = client["spbi"]

# Verificar la conexion con un ping
client.admin.command("ping")
print("Conexion a MongoDB Atlas establecida correctamente")
print("Base de datos activa:", db.name)

Conexion a MongoDB Atlas establecida correctamente
Base de datos activa: spbi


In [ ]:
# Paso 3 — Definir los 5 tickers mineros del proyecto
TICKERS = {
    "FSM": "Fortuna Silver Mines Inc.",
    "VOLCABC1.LM": "Volcan Compania Minera S.A.A.",
    "ABX.TO": "Barrick Gold Corporation",
    "BVN": "Compania de Minas Buenaventura",
    "BHP": "BHP Group Limited",
}

print("Tickers del proyecto:")
for ticker, nombre in TICKERS.items():
    print(f"  {ticker}: {nombre}")

Tickers del proyecto:
  FSM: Fortuna Silver Mines Inc.
  VOLCABC1.LM: Volcan Compania Minera S.A.A.
  ABX.TO: Barrick Gold Corporation
  BVN: Compania de Minas Buenaventura
  BHP: BHP Group Limited


In [ ]:
# Paso 4 — Importaciones y funciones de indicadores tecnicos
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime


def calcular_sma(serie, periodo):
    """Media movil simple (Simple Moving Average)."""
    return serie.rolling(window=periodo).mean()


def calcular_ema(serie, periodo):
    """Media movil exponencial (Exponential Moving Average)."""
    return serie.ewm(span=periodo, adjust=False).mean()


def calcular_rsi(serie, periodo=14):
    """Indice de Fuerza Relativa (Relative Strength Index)."""
    delta = serie.diff()
    ganancia = delta.where(delta > 0, 0).rolling(window=periodo).mean()
    perdida = -delta.where(delta < 0, 0).rolling(window=periodo).mean()
    rs = ganancia / perdida
    return 100 - (100 / (1 + rs))


print("Funciones de indicadores definidas: SMA, EMA, RSI")

Funciones de indicadores definidas: SMA, EMA, RSI


In [ ]:
# Paso 5 — Funcion de ingesta para un ticker individual
def limpiar_valor(v):
    """Convierte NaN en None para que el documento sea JSON/BSON valido."""
    return None if pd.isna(v) else round(float(v), 4)


def ingestar_ticker(ticker, periodo="1y"):
    """Descarga datos OHLCV reales, calcula indicadores y los guarda en MongoDB."""

    # Descarga de datos reales desde Yahoo Finance
    df = yf.download(ticker, period=periodo, auto_adjust=True, progress=False)

    if df.empty:
        print(f"  [{ticker}] Sin datos, se omite.")
        return 0

    # yfinance devuelve columnas con MultiIndex incluso para un solo ticker;
    # se aplanan tomando el primer nivel (Open, High, Low, Close, Volume)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # Calculo de indicadores tecnicos sobre el precio de cierre
    df["sma_20"] = calcular_sma(df["Close"], 20)
    df["sma_50"] = calcular_sma(df["Close"], 50)
    df["ema_12"] = calcular_ema(df["Close"], 12)
    df["ema_26"] = calcular_ema(df["Close"], 26)
    df["rsi_14"] = calcular_rsi(df["Close"], 14)

    # Conversion de cada fila del DataFrame a un documento de MongoDB
    registros = []
    for fecha, fila in df.iterrows():
        registros.append({
            "ticker": ticker,
            "fecha": fecha.strftime("%Y-%m-%d"),
            "open": limpiar_valor(fila["Open"]),
            "high": limpiar_valor(fila["High"]),
            "low": limpiar_valor(fila["Low"]),
            "close": limpiar_valor(fila["Close"]),
            "volume": int(fila["Volume"]) if not pd.isna(fila["Volume"]) else 0,
            "sma_20": limpiar_valor(fila["sma_20"]),
            "sma_50": limpiar_valor(fila["sma_50"]),
            "ema_12": limpiar_valor(fila["ema_12"]),
            "ema_26": limpiar_valor(fila["ema_26"]),
            "rsi_14": limpiar_valor(fila["rsi_14"]),
            "created_at": datetime.now(),
        })

    # Se eliminan registros previos del ticker para evitar duplicados y se insertan los nuevos
    db["precios_ohlcv"].delete_many({"ticker": ticker})
    db["precios_ohlcv"].insert_many(registros)

    print(f"  [{ticker}] {len(registros)} dias guardados en MongoDB")
    return len(registros)


print("Funcion ingestar_ticker definida")

Funcion ingestar_ticker definida


In [ ]:
# Paso 6 — Ingestar los 5 tickers del proyecto
print("Iniciando ingesta de los 5 tickers...")
print()

total = 0
for ticker in TICKERS.keys():
    total += ingestar_ticker(ticker, periodo="1y")

print()
print(f"Ingesta completa: {total} documentos en total")
print("Tickers presentes en la coleccion:", db["precios_ohlcv"].distinct("ticker"))

Iniciando ingesta de los 5 tickers...

  [FSM] 251 dias guardados en MongoDB
  [VOLCABC1.LM] 247 dias guardados en MongoDB
  [ABX.TO] 252 dias guardados en MongoDB
  [BVN] 251 dias guardados en MongoDB
  [BHP] 251 dias guardados en MongoDB

Ingesta completa: 1252 documentos en total
Tickers presentes en la coleccion: ['ABX.TO', 'BHP', 'BVN', 'FSM', 'VOLCABC1.LM']


In [ ]:
# Paso 7 — Celda de verificacion final
print("Resumen de la coleccion 'precios_ohlcv':")
print()

for ticker, nombre in TICKERS.items():
    n = db["precios_ohlcv"].count_documents({"ticker": ticker})
    ultimo = db["precios_ohlcv"].find_one({"ticker": ticker}, sort=[("fecha", -1)])

    if ultimo and ultimo.get("close") is not None:
        cierre = ultimo["close"]
        fecha = ultimo["fecha"]
        rsi = ultimo.get("rsi_14")
        rsi_txt = f"{rsi:.2f}" if rsi is not None else "N/D"
        print(f"  {ticker:12s} ({nombre}): {n} dias | ultimo cierre: {cierre:.2f} ({fecha}) | RSI-14: {rsi_txt}")
    else:
        print(f"  {ticker:12s} ({nombre}): {n} dias | sin datos de cierre disponibles")

print()
total_docs = db["precios_ohlcv"].count_documents({})
print(f"Total de documentos en la coleccion: {total_docs}")

# Verificacion de que no hay valores incompatibles con JSON (NaN) almacenados
muestra = db["precios_ohlcv"].find_one({})
print()
print("Documento de muestra:")
print(muestra)

Resumen de la coleccion 'precios_ohlcv':

  FSM          (Fortuna Silver Mines Inc.): 251 dias | ultimo cierre: 8.72 (2026-07-02) | RSI-14: 51.66
  VOLCABC1.LM  (Volcan Compania Minera S.A.A.): 247 dias | ultimo cierre: 0.86 (2026-07-02) | RSI-14: 57.18
  ABX.TO       (Barrick Gold Corporation): 252 dias | ultimo cierre: 55.56 (2026-07-03) | RSI-14: 48.12
  BVN          (Compania de Minas Buenaventura): 251 dias | ultimo cierre: 29.72 (2026-07-02) | RSI-14: 38.95
  BHP          (BHP Group Limited): 251 dias | ultimo cierre: 83.33 (2026-07-02) | RSI-14: 39.07

Total de documentos en la coleccion: 1252

Documento de muestra:
{'_id': ObjectId('6a4832b37d919a101bbf32df'), 'ticker': 'FSM', 'fecha': '2025-07-03', 'open': 6.55, 'high': 6.67, 'low': 6.51, 'close': 6.61, 'volume': 8507200, 'sma_20': None, 'sma_50': None, 'ema_12': 6.61, 'ema_26': 6.61, 'rsi_14': None, 'created_at': datetime.datetime(2026, 7, 3, 22, 7, 47, 533000)}


## Resultado

La colección `precios_ohlcv` contiene datos reales de los 5 tickers mineros, con precios OHLCV e indicadores técnicos (SMA-20, SMA-50, EMA-12, EMA-26, RSI-14). Los valores no disponibles (NaN) se almacenan como `None`, garantizando documentos válidos para JSON/BSON.

**Pipeline:** Yahoo Finance (`yfinance`) → corrección de MultiIndex → indicadores técnicos → **MongoDB Atlas** ✓

Estos datos quedan disponibles para los siguientes módulos del sistema (entrenamiento de modelos y servicio vía API).